<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Amortization-of-Risk-Indicators/blob/main/Amortization_RemovalTimes_(Sequential).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# GPU SEQUENTIAL ABLATION — COMMON ENGINE + STEP 1
# Condensed | Resume/New run | 20 min recovery | Best models
# ============================================================

import os,time,json,pickle,random
import numpy as np, pandas as pd, torch, torch.nn as nn
from collections import Counter
from torch.utils.data import TensorDataset,DataLoader,WeightedRandomSampler

# ---------- SETTINGS ----------
SEED=1234; T_OBS=10.; BINS=40; NMAX=400; MIN_REMOVALS=4
BATCH=64; MAX_EPOCHS=300; PATIENCE=20; LR=1e-3; WD=1e-6; LAMBDA_RHO=1.
CKPT_SEC=20*60

assert torch.cuda.is_available(), "GPU NOT FOUND. Colab: Runtime > Change runtime type > T4 GPU."
DEVICE=torch.device("cuda")
torch.set_float32_matmul_precision("high")
print("GPU:",torch.cuda.get_device_name(0))

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT="/content/drive/MyDrive/SIRS_removal_MLP"
except: ROOT="./SIRS_removal_MLP"

DATA_DIR=f"{ROOT}/T10_B40_min4"
ABL_DIR=f"{ROOT}/T10_B40_ablation_final"
os.makedirs(ABL_DIR,exist_ok=True)

# ---------- RESUME / NEW ----------
ans=input("\n1 = Resume existing run\n2 = New run\nChoice: ").strip()
assert ans in {"1","2"}
RESUME=(ans=="1")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

def load(name):
    with open(f"{DATA_DIR}/{name}","rb") as f:return pickle.load(f)

train_records=load("train_R10000.pkl")
val_records=load("validation.pkl")
test_records=load("test.pkl")
print("Existing data:",len(train_records),len(val_records),len(test_records))

# ---------- FEATURES ----------
BW=T_OBS/BINS
def basic(r): return np.r_[np.asarray(r["hist"],np.float32),np.asarray(r["summaries"],np.float32)]

def temporal(r):
    h=np.asarray(r["hist"],np.float32); x=h*NMAX; tot=float(x.sum()); den=max(tot,1.)
    nb=lambda w:max(1,min(BINS,int(np.ceil(w/BW))))
    n1,n2,n5=nb(1),nb(2),nb(5)
    l1,l2,l5=float(x[-n1:].sum()),float(x[-n2:].sum()),float(x[-n5:].sum())
    r1,r2,r5=l1,l2/2,l5/5
    f2=float(x[:n2].sum()); mid=BINS//2; s=max(0,mid-n2//2); m2=float(x[s:s+n2].sum())
    recent=x[-n5:]; tt=np.arange(len(recent),dtype=np.float32)*BW
    slope=float(np.polyfit(tt,recent,1)[0]) if len(recent)>1 and recent.sum()>0 else 0.
    mids=(np.arange(BINS,dtype=np.float32)+.5)*BW
    if tot>0:
        center=float((mids*x).sum()/tot); sd=float(np.sqrt((((mids-center)**2)*x).sum()/tot))
    else:center=sd=0.
    return np.array([
        l1/NMAX,l2/NMAX,l5/NMAX,r1/NMAX,r2/NMAX,r5/NMAX,
        l1/den,l2/den,l5/den,f2/NMAX,m2/NMAX,
        (l2-f2)/(l2+f2+1),(l2-m2)/(l2+m2+1),slope/NMAX,
        np.log((r1+.25)/(r5+.25)),np.log((r2+.25)/(r5+.25)),
        center/T_OBS,sd/T_OBS],np.float32)

def rich(r):
    b=basic(r); h=np.asarray(r["hist"],np.float32); c=np.cumsum(h)
    shape=c/c[-1] if c[-1]>0 else np.zeros_like(c)
    return np.r_[b,c,shape,temporal(r)].astype(np.float32)

# ---------- FIXED WEIGHTS ----------
def rg(n):
    if n<=7:return "4_7"
    if n<=15:return "8_15"
    if n<=31:return "16_31"
    return "32_plus"

strata=[("i0_1" if r["i0"]==1 else "i0_gt1",rg(r["n_removals"])) for r in train_records]
cnt=Counter(strata)
w=np.array([cnt[s]**-.5 for s in strata]); w/=w.mean()
TRAIN_W=torch.tensor(w,dtype=torch.double)

def train_loader(ds,seed,epoch):
    g=torch.Generator().manual_seed(SEED+seed+1000*epoch)
    smp=WeightedRandomSampler(TRAIN_W,len(TRAIN_W),replacement=True,generator=g)
    return DataLoader(ds,batch_size=BATCH,sampler=smp,pin_memory=True)

def val_loader(ds): return DataLoader(ds,batch_size=BATCH,shuffle=False,pin_memory=True)

# ---------- MLP ----------
class MLP(nn.Module):
    def __init__(self,din,dout,out="linear"):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(din,256),nn.SiLU(),nn.Linear(256,256),nn.SiLU(),
            nn.Linear(256,256),nn.SiLU(),nn.Linear(256,128),nn.SiLU(),
            nn.Linear(128,64),nn.SiLU(),nn.Linear(64,dout))
        self.out=out
    def forward(self,x):
        z=self.net(x)
        return torch.sigmoid(z) if self.out=="sigmoid" else nn.functional.softplus(z) if self.out=="softplus" else z

def meta(R):
    return tuple(np.asarray([r[k] for r in R],np.int64) for k in ["N","i0","n_removals"])

def padp(R):
    P=np.zeros((len(R),NMAX+2),np.float32)
    for j,r in enumerate(R): P[j,:len(r["p"])]=r["p"]
    return P

def targets(R,mode):
    a=[]
    for r in R:
        m,v=float(r["mean_tau"]),float(r["var_tau"])
        d=v if mode=="var" else v/max(m*m,1e-12)
        a.append([np.log1p(m),np.log1p(d)])
    return np.asarray(a,np.float32)

def base(x,N,i0,size):
    return torch.cat([x,(N.float()/NMAX)[:,None],(i0.float()/N.float())[:,None]],1) if size else x

def rm(path):
    if os.path.exists(path): os.remove(path)

def save_ck(path,net,opt,sch,epoch,batch,best,bad,finished=False):
    tmp=path+".tmp"
    torch.save(dict(model=net.state_dict(),optimizer=opt.state_dict(),scheduler=sch.state_dict(),
                    epoch=epoch,batch=batch,best=float(best),bad=int(bad),finished=finished),tmp)
    os.replace(tmp,path)

# ---------- HAZARD ----------
def pred_dist(net,x,N,i0,size):
    B=x.shape[0]; L=NMAX+1; z=base(x,N,i0,size)
    c=torch.arange(L,device=DEVICE,dtype=torch.float32)/NMAX
    Z=z[:,None,:].expand(-1,L,-1); C=c[None,:,None].expand(B,-1,-1)
    h=net(torch.cat([Z,C],2).reshape(B*L,-1)).reshape(B,L); q=1-h
    sb=torch.cat([torch.ones(B,1,device=DEVICE),torch.cumprod(q[:,:-1],1)],1)
    ph=torch.zeros(B,NMAX+2,device=DEVICE); ph[:,:L]=h*sb
    rows=torch.arange(B,device=DEVICE); ph[rows,N+1]=torch.cumprod(q,1)[rows,N]
    mask=torch.arange(NMAX+2,device=DEVICE)[None,:]<=N[:,None]+1
    return ph*mask,mask

def h_loss(net,b,size):
    x,N,i0,p=[z.to(DEVICE,non_blocking=True) for z in b]
    ph,m=pred_dist(net,x,N,i0,size); Lp=(((ph-p)**2)*m).sum(1).mean()
    p0=p*m; ph0=ph*m
    tail=torch.flip(torch.cumsum(torch.flip(p0,[1]),1),[1])-p0
    th=torch.flip(torch.cumsum(torch.flip(ph0,[1]),1),[1])-ph0
    tm=torch.arange(NMAX+1,device=DEVICE)[None,:]<=N[:,None]
    Lr=((((th[:,:NMAX+1]-tail[:,:NMAX+1])**2)*tm).sum(1)/(N.float()+1)).mean()
    return Lp+LAMBDA_RHO*Lr

def train_hazard(Xtr,Xva,Ntr,Nva,Itr,Iva,Ptr,Pva,size,out,seed,reuse=None):
    net=MLP(Xtr.shape[1]+1+(2 if size else 0),1,"sigmoid").to(DEVICE)
    if reuse:
        p=f"{ABL_DIR}/{reuse}/hazard_best.pt"; net.load_state_dict(torch.load(p,map_location=DEVICE,weights_only=True))
        print("HazardNet reused:",reuse); net.eval(); return net

    tr=TensorDataset(torch.tensor(Xtr),torch.tensor(Ntr),torch.tensor(Itr),torch.tensor(Ptr))
    va=TensorDataset(torch.tensor(Xva),torch.tensor(Nva),torch.tensor(Iva),torch.tensor(Pva))
    vl=val_loader(va); opt=torch.optim.AdamW(net.parameters(),lr=LR,weight_decay=WD)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,factor=.5,patience=12)
    bp,cp=f"{out}/hazard_best.pt",f"{out}/hazard_recovery.pt"
    if not RESUME: rm(bp); rm(cp)

    ep=bi0=0; best=np.inf; bad=0
    if RESUME and os.path.exists(cp):
        ck=torch.load(cp,map_location=DEVICE,weights_only=False); net.load_state_dict(ck["model"])
        opt.load_state_dict(ck["optimizer"]); sch.load_state_dict(ck["scheduler"])
        ep,bi0,best,bad=ck["epoch"],ck["batch"],ck["best"],ck["bad"]
        if ck.get("finished",False):
            net.load_state_dict(torch.load(bp,map_location=DEVICE,weights_only=True)); net.eval()
            print("HazardNet already complete."); return net
        print(f"HazardNet resume: epoch {ep+1}, batch {bi0+1}")

    last=time.time()
    while ep<MAX_EPOCHS:
        t=time.time(); net.train(); ls=[]; loader=train_loader(tr,seed,ep)
        for bi,b in enumerate(loader):
            if bi<bi0: continue
            opt.zero_grad(); L=h_loss(net,b,size); L.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(),5); opt.step(); ls.append(L.item())
            if time.time()-last>=CKPT_SEC:
                save_ck(cp,net,opt,sch,ep,bi+1,best,bad); last=time.time()
                print(f"  recovery saved: H epoch {ep+1}, batch {bi+1}",flush=True)
        bi0=0; net.eval()
        with torch.no_grad(): val=np.mean([h_loss(net,b,size).item() for b in vl])
        sch.step(val); imp=val<best-1e-6
        if imp: best=val; bad=0; torch.save(net.state_dict(),bp)
        else: bad+=1
        ep+=1; save_ck(cp,net,opt,sch,ep,0,best,bad); last=time.time()
        print(f"H {ep:3d}/{MAX_EPOCHS} train={np.mean(ls):.6f} val={val:.6f} best={best:.6f} "
              f"pat={bad}/{PATIENCE} lr={opt.param_groups[0]['lr']:.1e} time={time.time()-t:.1f}s"
              +(" BEST" if imp else ""),flush=True)
        if bad>=PATIENCE: break

    save_ck(cp,net,opt,sch,ep,0,best,bad,True)
    net.load_state_dict(torch.load(bp,map_location=DEVICE,weights_only=True)); net.eval(); return net

# ---------- HAZARD SUMMARIES ----------
SAFE_C=40
@torch.no_grad()
def h_summary(net,x,N,i0,size):
    B=x.shape[0]; L=SAFE_C+1; z=base(x,N,i0,size)
    c=torch.arange(L,device=DEVICE,dtype=torch.float32)/NMAX
    h=net(torch.cat([z[:,None,:].expand(-1,L,-1),c[None,:,None].expand(B,-1,-1)],2).reshape(B*L,-1)).reshape(B,L)
    q=1-h; sb=torch.cat([torch.ones(B,1,device=DEVICE),torch.cumprod(q[:,:-1],1)],1)
    mass=h*sb; tail=torch.cumprod(q,1); mc=tail[:,:SAFE_C].sum(1)
    k=torch.arange(SAFE_C,device=DEVICE,dtype=torch.float32)
    sd=torch.sqrt(torch.clamp((tail[:,:SAFE_C]*(2*k+1)[None,:]).sum(1)-mc**2,min=0)+1e-12)
    cap=torch.cat([mass[:,:SAFE_C],sb[:,SAFE_C:SAFE_C+1]],1); cap/=cap.sum(1,keepdim=True).clamp_min(1e-12)
    ent=-(cap*torch.log(cap+1e-12)).sum(1)/np.log(SAFE_C+1); cdf=torch.cumsum(cap,1)
    qs=[((cdf>=q).float().argmax(1).float()/SAFE_C) for q in [.25,.5,.75]]
    slope=(torch.log(tail[:,40]+1e-8)-torch.log(tail[:,10]+1e-8))/30
    return torch.stack([mass[:,0],tail[:,5],tail[:,10],tail[:,20],tail[:,30],tail[:,40],
        mc/SAFE_C,sd/SAFE_C,ent,*qs,tail.mean(1),tail[:,:11].mean(1),
        tail[:,10:21].mean(1),tail[:,20:41].mean(1),slope,h.mean(1)],1)

def calc_H(net,X,N,i0,size):
    ds=TensorDataset(torch.tensor(X),torch.tensor(N),torch.tensor(i0)); out=[]
    for x,n,j in val_loader(ds):
        out.append(h_summary(net,x.to(DEVICE),n.to(DEVICE),j.to(DEVICE),size).cpu().numpy())
    return np.concatenate(out)

# ---------- TAU ----------
def train_tau(Xtr,Xva,Ntr,Nva,Itr,Iva,Ytr0,Yva0,size,wm,wd,mode,out,seed):
    ym,ys=Ytr0.mean(0).astype(np.float32),Ytr0.std(0).astype(np.float32); ys[ys<1e-6]=1
    Ytr=((Ytr0-ym)/ys).astype(np.float32); Yva=((Yva0-ym)/ys).astype(np.float32)
    YM=torch.tensor(ym,device=DEVICE); YS=torch.tensor(ys,device=DEVICE)
    net=MLP(Xtr.shape[1]+(2 if size else 0),2,"softplus").to(DEVICE)
    tr=TensorDataset(torch.tensor(Xtr),torch.tensor(Ntr),torch.tensor(Itr),torch.tensor(Ytr))
    va=TensorDataset(torch.tensor(Xva),torch.tensor(Nva),torch.tensor(Iva),torch.tensor(Yva))
    vl=val_loader(va); opt=torch.optim.AdamW(net.parameters(),lr=LR,weight_decay=WD)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,factor=.5,patience=12)
    bp,cp=f"{out}/tau_best.pt",f"{out}/tau_recovery.pt"
    if not RESUME: rm(bp); rm(cp)

    def loss(b):
        x,N,i0,y=[z.to(DEVICE,non_blocking=True) for z in b]
        raw=net(base(x,N,i0,size)); e=(raw-YM)/YS-y
        return (wm*e[:,0]**2+wd*e[:,1]**2).mean()

    @torch.no_grad()
    def validate():
        net.eval(); ll=[]; em=[]; ed=[]
        for b in vl:
            x,N,i0,y=[z.to(DEVICE,non_blocking=True) for z in b]
            raw=net(base(x,N,i0,size)); e=(raw-YM)/YS-y
            ll.append((wm*e[:,0]**2+wd*e[:,1]**2).mean().item())
            truth=y*YS+YM
            mt,mp=torch.expm1(truth[:,0]),torch.expm1(raw[:,0]).clamp_min(0)
            dt,dp=torch.expm1(truth[:,1]),torch.expm1(raw[:,1]).clamp_min(0)
            em.append((abs(mp-mt)/mt.clamp_min(1e-12)).cpu().numpy())
            ed.append((abs(dp-dt)/dt.clamp_min(1e-12)).cpu().numpy())
        mm=float(np.median(np.concatenate(em))); md=float(np.median(np.concatenate(ed))); vl0=float(np.mean(ll))
        return vl0,mm,md,mm+.20*md if mode=="metric" else vl0

    ep=bi0=0; best=np.inf; bad=0
    if RESUME and os.path.exists(cp):
        ck=torch.load(cp,map_location=DEVICE,weights_only=False); net.load_state_dict(ck["model"])
        opt.load_state_dict(ck["optimizer"]); sch.load_state_dict(ck["scheduler"])
        ep,bi0,best,bad=ck["epoch"],ck["batch"],ck["best"],ck["bad"]
        if ck.get("finished",False):
            net.load_state_dict(torch.load(bp,map_location=DEVICE,weights_only=True))
            print("TauNet already complete."); return net,ym,ys
        print(f"TauNet resume: epoch {ep+1}, batch {bi0+1}")

    last=time.time()
    while ep<MAX_EPOCHS:
        t=time.time(); net.train(); ls=[]; loader=train_loader(tr,seed,ep)
        for bi,b in enumerate(loader):
            if bi<bi0: continue
            opt.zero_grad(); L=loss(b); L.backward(); torch.nn.utils.clip_grad_norm_(net.parameters(),5); opt.step()
            ls.append(L.item())
            if time.time()-last>=CKPT_SEC:
                save_ck(cp,net,opt,sch,ep,bi+1,best,bad); last=time.time()
                print(f"  recovery saved: T epoch {ep+1}, batch {bi+1}",flush=True)
        bi0=0; vl0,mm,md,score=validate(); sch.step(score); imp=score<best-1e-6
        if imp: best=score; bad=0; torch.save(net.state_dict(),bp)
        else: bad+=1
        ep+=1; save_ck(cp,net,opt,sch,ep,0,best,bad); last=time.time()
        print(f"T {ep:3d}/{MAX_EPOCHS} train={np.mean(ls):.6f} val={vl0:.6f} "
              f"mean={mm:.4f} disp={md:.4f} score={score:.4f} best={best:.4f} "
              f"pat={bad}/{PATIENCE} time={time.time()-t:.1f}s"+(" BEST" if imp else ""),flush=True)
        if bad>=PATIENCE: break

    save_ck(cp,net,opt,sch,ep,0,best,bad,True)
    net.load_state_dict(torch.load(bp,map_location=DEVICE,weights_only=True))
    return net,ym,ys

# ---------- TEST ----------
@torch.no_grad()
def evaluate(step,hnet,tnet,Xh,Xt,N,i0,nr,R,ym,ys,target_mode,size):
    PH=[]
    ds=TensorDataset(torch.tensor(Xh),torch.tensor(N),torch.tensor(i0))
    for x,n,j in val_loader(ds):
        PH.append(pred_dist(hnet,x.to(DEVICE),n.to(DEVICE),j.to(DEVICE),size)[0].cpu().numpy())
    PH=np.concatenate(PH)

    RAW=[]
    ds=TensorDataset(torch.tensor(Xt),torch.tensor(N),torch.tensor(i0))
    for x,n,j in val_loader(ds):
        RAW.append(tnet(base(x.to(DEVICE),n.to(DEVICE),j.to(DEVICE),size)).cpu().numpy())
    RAW=np.concatenate(RAW)

    pm=np.expm1(np.maximum(RAW[:,0],0))
    if target_mode=="var":
        pv=np.expm1(np.maximum(RAW[:,1],0)); pc=pv/np.maximum(pm**2,1e-12)
    else:
        pc=np.expm1(np.maximum(RAW[:,1],0)); pv=pc*pm**2

    rows=[]
    for j,r in enumerate(R):
        n=int(N[j]); pt=np.asarray(r["p"],float); pp=PH[j,:n+2]
        rt=np.flip(np.cumsum(np.flip(pt)))-pt; rp=np.flip(np.cumsum(np.flip(pp)))-pp; nz=pt>0
        mt,vt=float(r["mean_tau"]),float(r["var_tau"]); ct=vt/max(mt**2,1e-12)
        rows.append(dict(step=step,case_id=j,N=n,i0=int(i0[j]),n_removals=int(nr[j]),
            E2=float(np.linalg.norm(pp-pt)),Erho=float(np.max(abs(rp[:n+1]-rt[:n+1]))),
            Eover=float(abs(pp[n+1]-pt[n+1])),
            DKL=float(np.sum(pt[nz]*np.log(pt[nz]/np.clip(pp[nz],1e-12,None)))),
            RelMeanTau=abs(pm[j]-mt)/max(mt,1e-12),
            RelCV2Tau=abs(pc[j]-ct)/max(ct,1e-12),
            RelVarTau=abs(pv[j]-vt)/max(vt,1e-12)))
    return pd.DataFrame(rows)

# ---------- EXPERIMENT ----------
def run_experiment(step,rich_hist=False,h2t=False,target="var",wm=1.,wd=1.,ck_mode="loss",size=False,reuse=None):
    print("\n"+"="*90+"\n"+step+"\n"+"="*90)
    out=f"{ABL_DIR}/{step}"; os.makedirs(out,exist_ok=True)

    # New run clears only this step. Step 2 also clears its Hazard summary cache.
    if not RESUME:
        for f in ["hazard_best.pt","hazard_recovery.pt","tau_best.pt","tau_recovery.pt",
                  "H_train.npy","H_val.npy","H_test.npy"]:
            rm(f"{out}/{f}")

    fun=rich if rich_hist else basic
    A=[np.stack([fun(r) for r in R]) for R in [train_records,val_records,test_records]]
    xm,xs=A[0].mean(0).astype(np.float32),A[0].std(0).astype(np.float32); xs[xs<1e-6]=1
    Xtr,Xva,Xte=[((x-xm)/xs).astype(np.float32) for x in A]
    Ntr,Itr,Rtr=meta(train_records); Nva,Iva,Rva=meta(val_records); Nte,Ite,Rte=meta(test_records)

    print("[1/4] HazardNet")
    hnet=train_hazard(Xtr,Xva,Ntr,Nva,Itr,Iva,padp(train_records),padp(val_records),size,out,100,reuse)

    if h2t:
        print("[2/4] Hazard summaries")
        cache=f"{ABL_DIR}/{reuse}" if reuse else out
        files=[f"{cache}/H_{s}.npy" for s in ["train","val","test"]]
        if all(os.path.exists(f) for f in files): Htr,Hva,Hte=[np.load(f) for f in files]
        else:
            Htr,Hva,Hte=calc_H(hnet,Xtr,Ntr,Itr,size),calc_H(hnet,Xva,Nva,Iva,size),calc_H(hnet,Xte,Nte,Ite,size)
            [np.save(f,h) for f,h in zip(files,[Htr,Hva,Hte])]
        hm,hs=Htr.mean(0).astype(np.float32),Htr.std(0).astype(np.float32); hs[hs<1e-6]=1
        Htr,Hva,Hte=[((h-hm)/hs).astype(np.float32) for h in [Htr,Hva,Hte]]
        Ttr,Tva,Tte=[np.c_[x,h] for x,h in [(Xtr,Htr),(Xva,Hva),(Xte,Hte)]]
    else:Ttr,Tva,Tte=Xtr,Xva,Xte

    print("[3/4] TauNet")
    tnet,ym,ys=train_tau(Ttr,Tva,Ntr,Nva,Itr,Iva,targets(train_records,target),targets(val_records,target),
                         size,wm,wd,ck_mode,out,200)

    print("[4/4] Test")
    D=evaluate(step,hnet,tnet,Xte,Tte,Nte,Ite,Rte,test_records,ym,ys,target,size)
    D.to_csv(f"{out}/test_results_all.csv",index=False)
    E=D[D.n_removals>=MIN_REMOVALS]; E.to_csv(f"{out}/test_results_min4.csv",index=False)

    M=["E2","Erho","Eover","DKL","RelMeanTau","RelCV2Tau","RelVarTau"]
    S=E[M].quantile([.5,.25,.75,.95]); S.index=["median","Q1","Q3","P95"]; S.to_csv(f"{out}/summary_min4.csv")
    row=pd.DataFrame([{"step":step,**S.loc["median"].to_dict()}])
    gp=f"{ABL_DIR}/ablation_summary.csv"
    if os.path.exists(gp):
        old=pd.read_csv(gp); row=pd.concat([old[old.step!=step],row],ignore_index=True)
    row.to_csv(gp,index=False)
    print("\n",S.round(4)); return D

# ============================================================
# STEP 1: COMPLEX MLP BASELINE
# ============================================================
R1=run_experiment("01_baseline_complex",rich_hist=False,h2t=False,target="var",
                  wm=1.,wd=1.,ck_mode="loss",size=False)

GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

1 = Resume existing run
2 = New run
Choice: 1
Existing data: 10000 1000 1000

01_baseline_complex
[1/4] HazardNet
HazardNet already complete.
[3/4] TauNet
TauNet already complete.
[4/4] Test

             E2    Erho   Eover     DKL  RelMeanTau  RelCV2Tau  RelVarTau
median  0.0096  0.0151  0.0030  0.0083      0.1208     0.1911     0.2288
Q1      0.0044  0.0053  0.0002  0.0026      0.0556     0.0820     0.1010
Q3      0.0159  0.0290  0.0082  0.0204      0.2266     0.3678     0.5109
P95     0.0310  0.0771  0.0245  0.1255      0.4746     0.8658     1.3525


In [10]:
R2=run_experiment(
    "02_rich_history",
    rich_hist=True,
    h2t=False,
    target="var",
    wm=1.,wd=1.,
    ck_mode="loss",
    size=False
)


02_rich_history
[1/4] HazardNet
HazardNet already complete.
[3/4] TauNet
TauNet already complete.
[4/4] Test

             E2    Erho   Eover     DKL  RelMeanTau  RelCV2Tau  RelVarTau
median  0.0132  0.0218  0.0040  0.0120      0.0871     0.2143     0.1839
Q1      0.0053  0.0058  0.0002  0.0036      0.0362     0.0811     0.0819
Q3      0.0224  0.0406  0.0120  0.0261      0.2061     0.4162     0.4758
P95     0.0509  0.0829  0.0355  0.1132      0.5963     1.0662     1.8884


In [11]:
R3=run_experiment(
    "03_hazard_to_tau",
    rich_hist=True,
    h2t=True,
    target="var",
    wm=1.,wd=1.,
    ck_mode="loss",
    size=False,
    reuse="02_rich_history"
)


03_hazard_to_tau
[1/4] HazardNet
HazardNet reused: 02_rich_history
[2/4] Hazard summaries
[3/4] TauNet
TauNet already complete.
[4/4] Test

             E2    Erho   Eover     DKL  RelMeanTau  RelCV2Tau  RelVarTau
median  0.0132  0.0218  0.0040  0.0120      0.0793     0.1301     0.1643
Q1      0.0053  0.0058  0.0002  0.0036      0.0334     0.0601     0.0730
Q3      0.0224  0.0406  0.0120  0.0261      0.1515     0.2416     0.3143
P95     0.0509  0.0829  0.0355  0.1132      0.3118     0.5138     0.7765


In [12]:
R4=run_experiment(
    "04_CV2_parameterization",
    rich_hist=True,
    h2t=True,
    target="cv2",
    wm=1.,wd=1.,
    ck_mode="loss",
    size=False,
    reuse="02_rich_history"
)


04_CV2_parameterization
[1/4] HazardNet
HazardNet reused: 02_rich_history
[2/4] Hazard summaries
[3/4] TauNet
TauNet already complete.
[4/4] Test

             E2    Erho   Eover     DKL  RelMeanTau  RelCV2Tau  RelVarTau
median  0.0132  0.0218  0.0040  0.0120      0.0893     0.0070     0.1792
Q1      0.0053  0.0058  0.0002  0.0036      0.0360     0.0029     0.0720
Q3      0.0224  0.0406  0.0120  0.0261      0.1717     0.0160     0.3347
P95     0.0509  0.0829  0.0355  0.1132      0.3361     0.0887     0.7315


In [13]:
R5=run_experiment(
    "05_mean_focused_loss",
    rich_hist=True,
    h2t=True,
    target="cv2",
    wm=3.,wd=.5,
    ck_mode="loss",
    size=False,
    reuse="02_rich_history"
)


05_mean_focused_loss
[1/4] HazardNet
HazardNet reused: 02_rich_history
[2/4] Hazard summaries
[3/4] TauNet
TauNet already complete.
[4/4] Test

             E2    Erho   Eover     DKL  RelMeanTau  RelCV2Tau  RelVarTau
median  0.0132  0.0218  0.0040  0.0120      0.0348     0.0083     0.0692
Q1      0.0053  0.0058  0.0002  0.0036      0.0127     0.0026     0.0241
Q3      0.0224  0.0406  0.0120  0.0261      0.1046     0.0259     0.2162
P95     0.0509  0.0829  0.0355  0.1132      0.2949     0.1833     0.6083


In [14]:
R6=run_experiment(
    "06_metric_checkpoint",
    rich_hist=True,
    h2t=True,
    target="cv2",
    wm=3.,wd=.5,
    ck_mode="metric",
    size=False,
    reuse="02_rich_history"
)


06_metric_checkpoint
[1/4] HazardNet
HazardNet reused: 02_rich_history
[2/4] Hazard summaries
[3/4] TauNet
TauNet already complete.
[4/4] Test

             E2    Erho   Eover     DKL  RelMeanTau  RelCV2Tau  RelVarTau
median  0.0132  0.0218  0.0040  0.0120      0.0173     0.0070     0.0340
Q1      0.0053  0.0058  0.0002  0.0036      0.0068     0.0017     0.0131
Q3      0.0224  0.0406  0.0120  0.0261      0.0971     0.0218     0.1833
P95     0.0509  0.0829  0.0355  0.1132      0.2808     0.1529     0.5411


In [15]:
R7=run_experiment(
    "07_final_N_i0",
    rich_hist=True,
    h2t=True,
    target="cv2",
    wm=3.,wd=.5,
    ck_mode="metric",
    size=True
)

print("\nFINAL ABLATION TABLE")
print(pd.read_csv(f"{ABL_DIR}/ablation_summary.csv").round(4).to_string(index=False))


07_final_N_i0
[1/4] HazardNet
HazardNet already complete.
[2/4] Hazard summaries
[3/4] TauNet
TauNet resume: epoch 156, batch 1
T 156/300 train=0.002456 val=0.003436 mean=0.1086 disp=0.0188 score=0.1123 best=0.1123 pat=0/20 time=0.8s BEST
T 157/300 train=0.002342 val=0.003197 mean=0.0996 disp=0.0178 score=0.1032 best=0.1032 pat=0/20 time=0.8s BEST
T 158/300 train=0.002301 val=0.003496 mean=0.1091 disp=0.0191 score=0.1129 best=0.1032 pat=1/20 time=0.7s
T 159/300 train=0.002401 val=0.003034 mean=0.1026 disp=0.0205 score=0.1067 best=0.1032 pat=2/20 time=0.7s
T 160/300 train=0.002115 val=0.002905 mean=0.0937 disp=0.0170 score=0.0971 best=0.0971 pat=0/20 time=0.8s BEST
T 161/300 train=0.002241 val=0.003090 mean=0.0944 disp=0.0187 score=0.0981 best=0.0971 pat=1/20 time=0.7s
T 162/300 train=0.002077 val=0.002888 mean=0.0966 disp=0.0180 score=0.1002 best=0.0971 pat=2/20 time=0.7s
T 163/300 train=0.002131 val=0.002699 mean=0.0857 disp=0.0194 score=0.0896 best=0.0896 pat=0/20 time=0.8s BEST
T 1